# Deep Learning Diagnostics Lab — CIFAR-10 + TensorBoard + Representation Geometry

이 노트북은 작은 CIFAR-10 CNN에서 여러 진단법을 **한 학습 파이프라인 안에서 측정·시각화·저장**합니다.

## 핵심 원칙

- **Google Drive에 저장**: CSV / JSON / NPZ / PNG / TensorBoard logs / 분석 summary
- **Google Drive에 저장하지 않음**: model weights / checkpoints
- model weights는 Colab runtime의 `/content/local_checkpoints`에만 저장
- representation 분석은 같은 activation을 여러 관점으로 봄:
  - spectrum / effective rank
  - linear probe
  - CKA
  - PCA
  - UMAP
  - local PCA / kNN neighborhood
- optimizer dynamics:
  - gradient norm
  - update-to-weight ratio
- local geometry:
  - Hessian top eigenvalue
  - weight interpolation barrier

> 목표는 “그림을 예쁘게 그리는 것”이 아니라 **각 기법이 무엇을 측정하는지 결과 그림에서 바로 드러나게 하는 것**입니다.

## 0. Setup and storage policy

Drive 아래에는 다음만 저장됩니다.

```text
MyDrive/deep_learning_diagnostics/
├─ csv/
├─ npz/
├─ figures/
├─ tensorboard/
└─ summaries/
```

모델 weight/checkpoint는 오직:

```text
/content/local_checkpoints/
```

에 저장됩니다.

In [ ]:
!pip -q install umap-learn tensorboard

import os, json, math, random, time, shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torch.utils.tensorboard import SummaryWriter

from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import umap.umap_ as umap

from google.colab import drive

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 7
FAST_MODE = True

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/deep_learning_diagnostics")
CSV_DIR = DRIVE_ROOT / "csv"
NPZ_DIR = DRIVE_ROOT / "npz"
FIG_DIR = DRIVE_ROOT / "figures"
TB_DIR = DRIVE_ROOT / "tensorboard"
SUMMARY_DIR = DRIVE_ROOT / "summaries"

LOCAL_CKPT_DIR = Path("/content/local_checkpoints")  # weights stay local only

for d in [CSV_DIR, NPZ_DIR, FIG_DIR, TB_DIR, SUMMARY_DIR, LOCAL_CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Drive analysis root:", DRIVE_ROOT)
print("Local-only checkpoint root:", LOCAL_CKPT_DIR)

## 1. Data

CIFAR-10을 사용합니다.

FAST mode:
- train 12,000
- validation 2,000
- 6 epochs
- batch size 256

더 선명한 결과가 필요하면 `FAST_MODE=False`로 바꾸세요.

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

eval_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

root = "/content/data"
train_full = datasets.CIFAR10(root=root, train=True, download=True, transform=train_tf)
train_eval_full = datasets.CIFAR10(root=root, train=True, download=False, transform=eval_tf)

g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(train_full), generator=g).tolist()

if FAST_MODE:
    n_train, n_val, epochs = 12000, 2000, 6
else:
    n_train, n_val, epochs = 40000, 5000, 12

train_idx = perm[:n_train]
val_idx = perm[n_train:n_train+n_val]

train_ds = Subset(train_full, train_idx)
train_eval_ds = Subset(train_eval_full, train_idx)
val_ds = Subset(train_eval_full, val_idx)

BATCH = 256
loader_kwargs = dict(batch_size=BATCH, num_workers=2, pin_memory=True, persistent_workers=True)

train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
train_eval_loader = DataLoader(train_eval_ds, shuffle=False, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)

print(f"train={len(train_ds)}, val={len(val_ds)}, epochs={epochs}")

## 2. Small CNN with named representation stages

각 layer representation을

\[
z_\ell:\mathcal X\to\mathbb R^{d_\ell}
\]

로 보고, CNN feature map은 spatial average pooling해서 sample × feature 행렬로 만듭니다.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
        )
        self.block1 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 192, 3, padding=1, bias=False),
            nn.BatchNorm2d(192),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.penultimate = nn.Linear(192, 128)
        self.head = nn.Linear(128, num_classes)

    def forward(self, x, return_features=False):
        feats = {}
        x = self.stem(x)
        feats["stem"] = x.mean(dim=(2, 3))
        x = self.block1(x)
        feats["block1"] = x.mean(dim=(2, 3))
        x = self.block2(x)
        feats["block2"] = x.mean(dim=(2, 3))
        x = self.block3(x).flatten(1)
        z = F.relu(self.penultimate(x))
        feats["penultimate"] = z
        logits = self.head(z)
        return (logits, feats) if return_features else logits

seed_everything()
base_model = SmallCNN()
initial_state = {k: v.detach().cpu().clone() for k, v in base_model.state_dict().items()}

layer_names = ["stem", "block1", "block2", "penultimate"]
tracked_params = {
    "stem": "stem.0.weight",
    "block1": "block1.0.weight",
    "block2": "block2.0.weight",
    "penultimate": "penultimate.weight",
    "head": "head.weight",
}

## 3. Train + TensorBoard + local-only checkpoints

TensorBoard 기록:
- train / validation loss
- train / validation accuracy
- gradient norm
- update-to-weight ratio
- weight histogram

checkpoints는 `/content/local_checkpoints`에만 저장됩니다.

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = total_correct = total_n = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        total_loss += F.cross_entropy(logits, y, reduction="sum").item()
        total_correct += (logits.argmax(1) == y).sum().item()
        total_n += y.numel()
    return total_loss / total_n, total_correct / total_n

def train_one(run_name, optimizer_name):
    model = SmallCNN().to(DEVICE)
    model.load_state_dict(initial_state)

    if optimizer_name == "sgd":
        opt = torch.optim.SGD(model.parameters(), lr=0.08, momentum=0.9, weight_decay=5e-4)
    else:
        opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=5e-4)

    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    writer = SummaryWriter(str(TB_DIR / run_name))

    history = []
    dynamics = []
    checkpoint_epochs = sorted(set([0, 1, max(2, epochs//2), epochs]))

    torch.save(model.state_dict(), LOCAL_CKPT_DIR / f"{run_name}_epoch0.pt")

    global_step = 0
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = correct = n = 0

        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            loss.backward()

            named = dict(model.named_parameters())
            before = {tag: named[pname].detach().clone() for tag, pname in tracked_params.items()}

            batch_dyn = {}
            for tag, pname in tracked_params.items():
                p = named[pname]
                gnorm = p.grad.detach().norm().item()
                batch_dyn[tag] = {"grad_norm": gnorm}

            opt.step()

            named = dict(model.named_parameters())
            for tag, pname in tracked_params.items():
                p = named[pname]
                delta = (p.detach() - before[tag]).norm().item()
                wnorm = p.detach().norm().item()
                ratio = delta / (wnorm + 1e-12)
                batch_dyn[tag]["update_to_weight"] = ratio

                writer.add_scalar(f"dynamics/{tag}_grad_norm", batch_dyn[tag]["grad_norm"], global_step)
                writer.add_scalar(f"dynamics/{tag}_update_to_weight", ratio, global_step)

            dynamics.append({
                "run": run_name, "epoch": epoch, "step": global_step,
                **{f"{k}_grad_norm": v["grad_norm"] for k, v in batch_dyn.items()},
                **{f"{k}_update_to_weight": v["update_to_weight"] for k, v in batch_dyn.items()},
            })

            running_loss += loss.item() * y.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            n += y.numel()
            global_step += 1

        sched.step()
        train_loss, train_acc = running_loss / n, correct / n
        val_loss, val_acc = evaluate(model, val_loader)

        row = dict(run=run_name, epoch=epoch, train_loss=train_loss, train_acc=train_acc,
                   val_loss=val_loss, val_acc=val_acc, lr=opt.param_groups[0]["lr"])
        history.append(row)

        for k, v in row.items():
            if k not in ("run", "epoch"):
                writer.add_scalar(f"epoch/{k}", v, epoch)

        for tag, pname in tracked_params.items():
            writer.add_histogram(f"weights/{tag}", dict(model.named_parameters())[pname], epoch)

        if epoch in checkpoint_epochs:
            torch.save(model.state_dict(), LOCAL_CKPT_DIR / f"{run_name}_epoch{epoch}.pt")

        print(f"{run_name:5s} e{epoch:02d}: train={train_loss:.3f}/{train_acc:.3f} "
              f"val={val_loss:.3f}/{val_acc:.3f}")

    writer.close()
    pd.DataFrame(history).to_csv(CSV_DIR / f"{run_name}_history.csv", index=False)
    pd.DataFrame(dynamics).to_csv(CSV_DIR / f"{run_name}_dynamics.csv", index=False)
    return model, pd.DataFrame(history), pd.DataFrame(dynamics), checkpoint_epochs

sgd_model, sgd_hist, sgd_dyn, sgd_ckpt_epochs = train_one("sgd", "sgd")
adam_model, adam_hist, adam_dyn, adam_ckpt_epochs = train_one("adamw", "adamw")

In [ ]:
# TensorBoard 실행
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/deep_learning_diagnostics/tensorboard

## 4. Save and visualize training curves / dynamics

In [ ]:
def savefig(name):
    path = FIG_DIR / name
    plt.savefig(path, dpi=170, bbox_inches="tight")
    print("saved:", path)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for hist, label in [(sgd_hist, "SGD"), (adam_hist, "AdamW")]:
    axes[0].plot(hist.epoch, hist.train_loss, marker="o", label=f"{label} train")
    axes[0].plot(hist.epoch, hist.val_loss, marker="o", ls="--", label=f"{label} val")
    axes[1].plot(hist.epoch, hist.train_acc, marker="o", label=f"{label} train")
    axes[1].plot(hist.epoch, hist.val_acc, marker="o", ls="--", label=f"{label} val")
axes[0].set_title("Loss")
axes[1].set_title("Accuracy")
axes[1].set_ylim(0, 1)
for ax in axes: ax.legend()
plt.tight_layout()
savefig("training_curves.png")
plt.show()

In [ ]:
def plot_dyn(df, run_name):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for tag in tracked_params:
        axes[0].plot(df[f"{tag}_grad_norm"].rolling(20, min_periods=1).mean(), label=tag)
        axes[1].plot(df[f"{tag}_update_to_weight"].rolling(20, min_periods=1).mean(), label=tag)
    axes[0].set_yscale("log")
    axes[1].set_yscale("log")
    axes[0].set_title(f"{run_name}: gradient norm")
    axes[1].set_title(f"{run_name}: update / weight ratio")
    for ax in axes: ax.legend()
    plt.tight_layout()
    savefig(f"{run_name}_dynamics.png")
    plt.show()

plot_dyn(sgd_dyn, "SGD")
plot_dyn(adam_dyn, "AdamW")

## 5. Feature extraction + Drive NPZ

representation 분석 데이터는 Drive의 NPZ에 저장합니다.

저장:
- activation matrices
- labels
- predicted labels
- correctness mask

weights는 저장하지 않습니다.

In [ ]:
@torch.no_grad()
def collect_features(model, loader, max_samples=3000):
    model.eval()
    all_feats = {k: [] for k in layer_names}
    ys, preds = [], []
    n = 0
    for x, y in loader:
        x = x.to(DEVICE)
        logits, feats = model(x, return_features=True)
        take = min(x.size(0), max_samples - n)
        for k in layer_names:
            all_feats[k].append(feats[k][:take].cpu())
        ys.append(y[:take].cpu())
        preds.append(logits[:take].argmax(1).cpu())
        n += take
        if n >= max_samples: break
    all_feats = {k: torch.cat(v).numpy() for k, v in all_feats.items()}
    ys = torch.cat(ys).numpy()
    preds = torch.cat(preds).numpy()
    return all_feats, ys, preds

sgd_feats, y_val, pred_sgd = collect_features(sgd_model, val_loader, 2000)
adam_feats, _, pred_adam = collect_features(adam_model, val_loader, 2000)

np.savez_compressed(
    NPZ_DIR / "sgd_val_representations.npz",
    y=y_val, pred=pred_sgd, **sgd_feats
)
np.savez_compressed(
    NPZ_DIR / "adamw_val_representations.npz",
    y=y_val, pred=pred_adam, **adam_feats
)
print("saved representation NPZ files")

## 6. Spectrum + effective rank

\[
p_i=\frac{s_i^2}{\sum_j s_j^2},
\qquad
r_{\mathrm{eff}}
=
\exp\left(-\sum_i p_i\log p_i\right)
\]

### 그림 해석
- spectrum이 빠르게 떨어짐 → 소수 방향에 representation energy가 집중
- effective rank가 낮음 → 실질적 사용 차원이 작음
- probe와 함께 봐야 compression인지 collapse인지 구분 가능

In [ ]:
def spectrum_erank(X):
    X = torch.tensor(X, dtype=torch.float32)
    X = X - X.mean(0, keepdim=True)
    s = torch.linalg.svdvals(X)
    power = s.square()
    p = power / power.sum().clamp_min(1e-12)
    er = torch.exp(-(p * torch.log(p.clamp_min(1e-12))).sum()).item()
    return s.numpy(), p.numpy(), er

rows = []
fig, axes = plt.subplots(1, len(layer_names), figsize=(16, 3.5))
for ax, layer in zip(axes, layer_names):
    s, p, er = spectrum_erank(sgd_feats[layer])
    rows.append({"layer": layer, "effective_rank": er})
    k = min(40, len(p))
    ax.plot(np.arange(1, k+1), p[:k], marker=".")
    ax.set_yscale("log")
    ax.set_title(f"{layer}\neff.rank={er:.1f}")
    ax.set_xlabel("spectral direction")
plt.tight_layout()
savefig("sgd_spectrum_effective_rank.png")
plt.show()

pd.DataFrame(rows).to_csv(CSV_DIR / "sgd_effective_rank.csv", index=False)

## 7. Linear probe

각 representation은 고정하고 선형 classifier만 학습합니다.

\[
h_\ell(z)=W_\ell z+b_\ell
\]

**목적:** class 정보가 해당 layer에서 얼마나 선형적으로 읽히는지 측정합니다.

In [ ]:
@torch.no_grad()
def collect_features_from_loader(model, loader, max_samples):
    return collect_features(model, loader, max_samples)

train_feats, train_y, _ = collect_features(sgd_model, train_eval_loader, 5000)

def fit_probe(Xtr, ytr, Xva, yva, steps=180, lr=0.08):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(Xtr).astype(np.float32)
    Xva = scaler.transform(Xva).astype(np.float32)
    Xtr = torch.tensor(Xtr, device=DEVICE)
    Xva = torch.tensor(Xva, device=DEVICE)
    ytr = torch.tensor(ytr, device=DEVICE)
    yva = torch.tensor(yva, device=DEVICE)

    clf = nn.Linear(Xtr.shape[1], 10).to(DEVICE)
    opt = torch.optim.AdamW(clf.parameters(), lr=lr, weight_decay=1e-3)
    for _ in range(steps):
        opt.zero_grad(set_to_none=True)
        loss = F.cross_entropy(clf(Xtr), ytr)
        loss.backward()
        opt.step()
    with torch.no_grad():
        return (clf(Xva).argmax(1) == yva).float().mean().item()

probe_rows = []
for layer in layer_names:
    acc = fit_probe(train_feats[layer], train_y, sgd_feats[layer], y_val)
    probe_rows.append({"layer": layer, "probe_accuracy": acc})

probe_df = pd.DataFrame(probe_rows)
probe_df.to_csv(CSV_DIR / "sgd_linear_probe.csv", index=False)

plt.figure(figsize=(7,4))
plt.bar(probe_df.layer, probe_df.probe_accuracy)
plt.ylim(0,1)
plt.ylabel("validation accuracy")
plt.title("Linear probe by representation layer")
savefig("sgd_linear_probe.png")
plt.show()

## 8. PCA / UMAP representation visualization

### PCA
전역적으로 큰 분산 방향을 보여줍니다.

### UMAP
local neighborhood가 2D에서 어떻게 배치되는지 보는 탐색적 시각화입니다.

두 그림 모두 **정답 class 색**과 **오분류 표시**를 같이 봅니다.

> PCA/UMAP 자체가 manifold의 증명은 아닙니다. point cloud의 전역/국소 구조를 보기 위한 시각화입니다.

In [ ]:
def reduce_and_save(X, y, pred, layer, method="pca"):
    Xs = StandardScaler().fit_transform(X)
    if method == "pca":
        reducer = PCA(n_components=2, random_state=SEED)
    else:
        reducer = umap.UMAP(n_components=2, n_neighbors=20, min_dist=0.15,
                            metric="euclidean", random_state=SEED)
    Z2 = reducer.fit_transform(Xs)

    df = pd.DataFrame({
        "x": Z2[:,0], "y": Z2[:,1],
        "label": y, "pred": pred,
        "correct": (y == pred)
    })
    df.to_csv(CSV_DIR / f"sgd_{layer}_{method}_2d.csv", index=False)

    plt.figure(figsize=(7,6))
    sc = plt.scatter(df.x, df.y, c=df.label, s=10, alpha=0.65, cmap="tab10")
    wrong = df[~df.correct]
    plt.scatter(wrong.x, wrong.y, facecolors="none", edgecolors="black", s=34, linewidths=0.8)
    plt.title(f"{method.upper()} — {layer}\nblack ring = misclassified")
    plt.xlabel(f"{method.upper()}-1")
    plt.ylabel(f"{method.upper()}-2")
    plt.colorbar(sc, ticks=range(10), label="CIFAR-10 class")
    savefig(f"sgd_{layer}_{method}.png")
    plt.show()
    return df

for layer in ["stem", "block2", "penultimate"]:
    reduce_and_save(sgd_feats[layer], y_val, pred_sgd, layer, "pca")
    reduce_and_save(sgd_feats[layer], y_val, pred_sgd, layer, "umap")

## 9. TensorBoard Embedding Projector

같은 sample들의 representation을 TensorBoard Projector에서 직접 회전/확대해서 볼 수 있습니다.

- metadata: ground-truth class / prediction / correctness
- 각 layer embedding 기록

In [ ]:
projector_logdir = TB_DIR / "projector_sgd"
writer = SummaryWriter(str(projector_logdir))

metadata = [f"y={int(y)} | pred={int(p)} | correct={bool(y==p)}"
            for y, p in zip(y_val, pred_sgd)]

for layer in layer_names:
    writer.add_embedding(
        torch.tensor(sgd_feats[layer], dtype=torch.float32),
        metadata=metadata,
        tag=f"sgd_{layer}"
    )
writer.close()

print("Embedding Projector logs saved to:", projector_logdir)
print("Re-run TensorBoard and open the Projector tab.")

## 10. Local manifold proxy — kNN + local PCA intrinsic dimension

각 sample \(z_i\) 주변의 \(k\)-nearest neighbors를 모아 local covariance를 계산합니다.

local PCA eigenvalue \(\lambda_1\ge\lambda_2\ge\cdots\)에 대해,
누적 설명분산 90%를 넘는 최소 차원을 **local PCA dimension proxy**로 씁니다.

\[
d_{\mathrm{local}}(z_i)
=
\min\left\{
m:
\frac{\sum_{j=1}^{m}\lambda_j}
{\sum_j\lambda_j}
\ge 0.9
\right\}
\]

이 값은 엄밀한 intrinsic dimension의 유일한 정의가 아니라, **국소적으로 몇 개 선형 방향이 필요한지** 보는 직관적 proxy입니다.

In [ ]:
def local_pca_dimensions(X, k=30, variance_threshold=0.90):
    Xs = StandardScaler().fit_transform(X)
    nbrs = NearestNeighbors(n_neighbors=k+1).fit(Xs)
    _, idx = nbrs.kneighbors(Xs)

    dims = []
    for i in range(len(Xs)):
        neigh = Xs[idx[i,1:]]
        neigh = neigh - neigh.mean(0, keepdims=True)
        s = np.linalg.svd(neigh, compute_uv=False)
        eig = s**2
        cum = np.cumsum(eig) / max(eig.sum(), 1e-12)
        dims.append(int(np.searchsorted(cum, variance_threshold) + 1))
    return np.array(dims), idx

local_dim, nn_idx = local_pca_dimensions(sgd_feats["penultimate"], k=30)

local_df = pd.DataFrame({
    "label": y_val,
    "pred": pred_sgd,
    "correct": y_val == pred_sgd,
    "local_pca_dim90": local_dim
})
local_df.to_csv(CSV_DIR / "sgd_penultimate_local_pca_dimension.csv", index=False)

plt.figure(figsize=(8,4))
for c in range(10):
    vals = local_dim[y_val == c]
    plt.hist(vals, bins=np.arange(local_dim.min(), local_dim.max()+2)-0.5,
             alpha=0.25, label=str(c))
plt.xlabel("local PCA dimension (90% variance)")
plt.ylabel("count")
plt.title("Local dimension distribution by class")
plt.legend(ncol=5)
savefig("sgd_local_pca_dimension_by_class.png")
plt.show()

plt.figure(figsize=(6,4))
data = [
    local_dim[y_val == pred_sgd],
    local_dim[y_val != pred_sgd]
]
plt.boxplot(data, tick_labels=["correct", "wrong"])
plt.ylabel("local PCA dimension (90% variance)")
plt.title("Local geometry around correct vs misclassified samples")
savefig("sgd_local_dim_correct_vs_wrong.png")
plt.show()

## 11. Visualize one local neighborhood + tangent directions

penultimate representation을 PCA 2D로 내려서,
한 sample의 kNN neighborhood를 강조하고 local PCA tangent directions뉼 그립니다.

**목적:** local PCA가 실제로 어느 주변 점구름의 방향 구조뉼 측정하는지 눈으로 확인합니다.

In [ ]:
X = StandardScaler().fit_transform(sgd_feats["penultimate"])
global_pca = PCA(n_components=2, random_state=SEED)
X2 = global_pca.fit_transform(X)

# 오분류 sample이 있으면 그것을 선택, 없으면 첫 sample
wrong_idx = np.where(y_val != pred_sgd)[0]
center_i = int(wrong_idx[0]) if len(wrong_idx) else 0
neighbors = nn_idx[center_i, 1:]

local = X[neighbors]
local_center = local.mean(0)
U, S, VT = np.linalg.svd(local - local_center, full_matrices=False)

# local top directions를 global PCA 2D 좌표에 projection
dirs2 = VT[:2] @ global_pca.components_.T
scale = 2.5

plt.figure(figsize=(7,6))
plt.scatter(X2[:,0], X2[:,1], c=y_val, s=8, alpha=0.18, cmap="tab10")
plt.scatter(X2[neighbors,0], X2[neighbors,1], c=y_val[neighbors], s=28, cmap="tab10")
plt.scatter(X2[center_i,0], X2[center_i,1], marker="*", s=220, c="black")

for j in range(2):
    dx, dy = dirs2[j] * scale
    plt.arrow(X2[center_i,0], X2[center_i,1], dx, dy,
              width=0.01, head_width=0.12, length_includes_head=True)

plt.title(f"Local neighborhood around sample {center_i}\n"
          f"true={y_val[center_i]}, pred={pred_sgd[center_i]}")
plt.xlabel("global PCA-1")
plt.ylabel("global PCA-2")
savefig("sgd_local_neighborhood_tangent_proxy.png")
plt.show()

## 12. CKA — SGD vs AdamW representation similarity

\[
\mathrm{CKA}(X,Y)
=
\frac{\|X^\top Y\|_F^2}
{\|X^\top X\|_F\,\|Y^\top Y\|_F}
\]

같은 입력 sample에 대한 두 representation의 구조적 유사도를 봅니다.

In [ ]:
def linear_cka(X, Y):
    X = torch.tensor(X, dtype=torch.float32)
    Y = torch.tensor(Y, dtype=torch.float32)
    X -= X.mean(0, keepdim=True)
    Y -= Y.mean(0, keepdim=True)
    return (
        (X.T @ Y).square().sum()
        / (((X.T @ X).square().sum().sqrt() * (Y.T @ Y).square().sum().sqrt()) + 1e-12)
    ).item()

cka_rows = []
for layer in layer_names:
    c = linear_cka(sgd_feats[layer], adam_feats[layer])
    cka_rows.append({"layer": layer, "cka_sgd_vs_adamw": c})

cka_df = pd.DataFrame(cka_rows)
cka_df.to_csv(CSV_DIR / "cka_sgd_vs_adamw.csv", index=False)

plt.figure(figsize=(7,4))
plt.bar(cka_df.layer, cka_df.cka_sgd_vs_adamw)
plt.ylim(0,1)
plt.ylabel("linear CKA")
plt.title("Same initialization, different optimizer")
savefig("cka_sgd_vs_adamw.png")
plt.show()

## 13. Hessian top eigenvalue

\[
H(\theta)=\nabla_\theta^2L(\theta)
\]

전체 Hessian은 만들지 않고 Hessian-vector product로 top eigenvalue를 추정합니다.

In [ ]:
def load_local_state(run, epoch):
    return torch.load(LOCAL_CKPT_DIR / f"{run}_epoch{epoch}.pt", map_location="cpu")

def hessian_top_eigenvalue(model_template, state_dict, loader, power_iters=10):
    m = SmallCNN().to(DEVICE)
    m.load_state_dict(state_dict)
    m.eval()
    x, y = next(iter(loader))
    x, y = x[:128].to(DEVICE), y[:128].to(DEVICE)
    params = [p for p in m.parameters() if p.requires_grad]

    def normalize(vs):
        norm = torch.sqrt(sum((v*v).sum() for v in vs)).clamp_min(1e-12)
        return [v/norm for v in vs]

    vs = normalize([torch.randn_like(p) for p in params])
    eig = np.nan
    for _ in range(power_iters):
        loss = F.cross_entropy(m(x), y)
        grads = torch.autograd.grad(loss, params, create_graph=True)
        gv = sum((g*v).sum() for g,v in zip(grads,vs))
        hvs = torch.autograd.grad(gv, params)
        eig = sum((v*hv).sum() for v,hv in zip(vs,hvs)).item()
        vs = normalize([hv.detach() for hv in hvs])
    return eig

hessian_rows = []
for label, run, epoch in [
    ("SGD init", "sgd", 0),
    ("SGD final", "sgd", epochs),
    ("AdamW final", "adamw", epochs),
]:
    val = hessian_top_eigenvalue(base_model, load_local_state(run, epoch), train_eval_loader)
    hessian_rows.append({"condition": label, "lambda_max": val})

hess_df = pd.DataFrame(hessian_rows)
hess_df.to_csv(CSV_DIR / "hessian_top_eigenvalue.csv", index=False)

plt.figure(figsize=(7,4))
plt.bar(hess_df.condition, hess_df.lambda_max)
plt.ylabel(r"estimated $\lambda_{\max}(H)$")
plt.title("Local curvature")
plt.xticks(rotation=15)
savefig("hessian_top_eigenvalue.png")
plt.show()

## 14. Weight interpolation barrier

\[
\theta(\alpha)=(1-\alpha)\theta_A+\alpha\theta_B,\qquad \alpha\in[0,1]
\]

두 solution 사이 직선 parameter path의 loss/accuracy 변화를 봅니다.

In [ ]:
def interpolate_state(a, b, alpha):
    out = {}
    for k in a:
        if torch.is_floating_point(a[k]):
            out[k] = (1-alpha)*a[k] + alpha*b[k]
        else:
            out[k] = a[k] if alpha < 0.5 else b[k]
    return out

@torch.no_grad()
def interpolation_curve(state_a, state_b, loader, alphas):
    m = SmallCNN().to(DEVICE)
    rows = []
    for a in alphas:
        m.load_state_dict(interpolate_state(state_a, state_b, float(a)))
        loss, acc = evaluate(m, loader)
        rows.append({"alpha": float(a), "loss": loss, "accuracy": acc})
    return pd.DataFrame(rows)

alphas = np.linspace(0,1,21)
mid_epoch = max(1, epochs//2)

same_df = interpolation_curve(
    load_local_state("sgd", mid_epoch),
    load_local_state("sgd", epochs),
    val_loader, alphas
)
cross_df = interpolation_curve(
    load_local_state("sgd", epochs),
    load_local_state("adamw", epochs),
    val_loader, alphas
)

same_df["path"] = f"SGD e{mid_epoch}→e{epochs}"
cross_df["path"] = "SGD final→AdamW final"
interp_df = pd.concat([same_df, cross_df], ignore_index=True)
interp_df.to_csv(CSV_DIR / "weight_interpolation.csv", index=False)

fig, axes = plt.subplots(1,2,figsize=(12,4))
for df, label in [(same_df, "same-run"), (cross_df, "cross-optimizer")]:
    axes[0].plot(df.alpha, df.loss, marker="o", label=label)
    axes[1].plot(df.alpha, df.accuracy, marker="o", label=label)
axes[0].set_title("Interpolation loss")
axes[1].set_title("Interpolation accuracy")
axes[1].set_ylim(0,1)
for ax in axes:
    ax.set_xlabel("alpha")
    ax.legend()
savefig("weight_interpolation.png")
plt.show()

## 15. Save experiment summary

모든 핵심 진단 수치를 하나의 JSON으로 저장합니다.

In [ ]:
summary = {
    "torch_version": torch.__version__,
    "device": str(DEVICE),
    "fast_mode": FAST_MODE,
    "train_size": len(train_ds),
    "val_size": len(val_ds),
    "epochs": epochs,
    "sgd_final_val_acc": float(sgd_hist.iloc[-1].val_acc),
    "adamw_final_val_acc": float(adam_hist.iloc[-1].val_acc),
    "sgd_effective_rank": {
        layer: float(spectrum_erank(sgd_feats[layer])[2]) for layer in layer_names
    },
    "sgd_probe_accuracy": dict(zip(probe_df.layer, probe_df.probe_accuracy.astype(float))),
    "cka_sgd_vs_adamw": dict(zip(cka_df.layer, cka_df.cka_sgd_vs_adamw.astype(float))),
    "hessian_top_eigenvalue": dict(zip(hess_df.condition, hess_df.lambda_max.astype(float))),
    "local_pca_dim90_mean": float(local_dim.mean()),
    "local_pca_dim90_correct_mean": float(local_dim[y_val == pred_sgd].mean()),
    "local_pca_dim90_wrong_mean": float(local_dim[y_val != pred_sgd].mean()) if np.any(y_val != pred_sgd) else None,
}

with open(SUMMARY_DIR / "experiment_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

## 16. Final file check

Drive에는 분석 결과만 있어야 합니다.

In [ ]:
print("=== Google Drive analysis artifacts ===")
for p in sorted(DRIVE_ROOT.rglob("*")):
    if p.is_file():
        print(p.relative_to(DRIVE_ROOT))

print("\n=== Local-only model checkpoints ===")
for p in sorted(LOCAL_CKPT_DIR.glob("*.pt")):
    print(p)

assert not any(DRIVE_ROOT.rglob("*.pt")), "Model weights unexpectedly found on Google Drive!"
print("\nStorage policy check passed: no .pt weights on Drive.")

## Reading order

이 노트북 결과는 다음 순서로 읽는 것을 권장합니다.

```text
loss / accuracy
→ gradient norm + update/weight
→ spectrum + effective rank
→ linear probe
→ PCA / UMAP
→ local PCA + kNN neighborhood
→ CKA
→ Hessian
→ interpolation barrier
```

핵심은 **같은 representation을 서로 다른 질문으로 여러 번 본다**는 것입니다.

- spectrum / rank: 몇 개 방향을 쓰는가?
- probe: label 정보가 읽히는가?
- PCA / UMAP: sample들이 어떻게 배치되는가?
- local PCA: 한 sample 주변에 몇 개 방향이 필요한가?
- CKA: 다른 학습 조건에서도 같은 구조인가?
- Hessian: parameter-space에서 얼마나 가파르게 휘는가?
- interpolation: 두 solution 사이에 barrier가 있는가?